In [1]:
### packages to import
import scyjava
import os
import numpy as np
import tifffile
from PIL import Image
import re


#entries to fill in
directory = "D:/Vaky/D14_and_PTB_samples/2026-09-29-samplesJuly_D14PTB/D14F_slide3_S5/mp1"
sampling = 5
cropx = 5 # en pixels, ce qu'il a a enlever de chaque cote, en sampling 1
cropy = 0
objective =  '20xZ_Water' # used only if XYZ. Values accepted: '25xO_Water', '20xZ_Water', '20xO_Air', '20xZ_Air'
mode = 'other' # indicate mode chosen. Values accepted: 'XYZ', 'other'.
sat_corr = 1 # 1 if want saturation correction, 0 otherwise
pix_rate = 200 # used only if saturation correction (kHz)


# code starts here: don't change anything

# Setting the pixel size (um) based on the objective chosen by the user, used only if mode XYZ (no need for FFT MATLAB processing) and want to scale the images directly 
if objective =='20xZ_Water':
    obj_pix=1.98 
elif objective =='25xO_Water':
    obj_pix=1.69
elif objective =='20xO_Air':
    obj_pix=2.11
elif objective =='20xZ_Air':
    obj_pix=1.95



#extracting from any image the height and width in pixels
for filename in os.listdir(directory):
    if filename.endswith(".tif"):
        image_path = os.path.join(directory, filename) 
        img = tifffile.imread(image_path)
        height = img.shape[0] 
        width = img.shape[1]
        break
        
#redefining the crop size in pixels given the sampling
cropx = cropx*sampling 
cropy = cropy*sampling

laserFq = 80000 #in kHz (the pulse rate of the laser, unchanged)

files = os.listdir(directory) 
if sat_corr==1: 
    os.makedirs(os.path.join(directory, "data_satcorr"), exist_ok=True)
else: 
    os.makedirs(os.path.join(directory, "data_nosat"), exist_ok=True)

pixel_size_um = obj_pix/sampling # microns per pixel

for filename in files:
    if filename.endswith(".tif"):
        image_path = os.path.join(directory, filename)
        img = tifffile.imread(image_path)    
        cropped_img = img[cropy:height-cropy, cropx:width-cropx]
        div_cropped_img = (cropped_img / 14.0).astype(np.float32)
        
        if sat_corr==1: 
            img_tmp = div_cropped_img # temporary image for saturation correction
            img_cor = np.zeros_like(img_tmp)
            sat = 1.5/5*laserFq/pix_rate
            mask = img_tmp < sat
            lasershots = laserFq/pix_rate
            img_cor[mask] = -np.log(1.0 - img_tmp[mask] / lasershots) * lasershots
            scaled_div_cropped_img = np.clip(img_cor, 0, 255).astype(np.uint8)
        
            savingpath = os.path.join(directory, "data_satcorr", filename)
        
            tifffile.imwrite(savingpath, scaled_div_cropped_img, imagej=True,
                resolution=(1/pixel_size_um, 1/pixel_size_um),
                metadata={
                    "unit": "um",
                    "spacing": pixel_size_um,  # for z if needed
                }) 
        else: 
            scaled_div_cropped_img = np.clip(div_cropped_img, 0, 255).astype(np.uint8)
            savingpath = os.path.join(directory, "data_nosat", filename)
            tifffile.imwrite(savingpath, scaled_div_cropped_img, imagej=True,
                resolution=(1/pixel_size_um, 1/pixel_size_um),
                metadata={
                    "unit": "um",
                    "spacing": pixel_size_um,  # for z if needed
                }) 

if mode == 'XYZ':

    # choose the correct subfolder
    if sat_corr == 1:
        data_dir = os.path.join(directory, "data_satcorr")
    else:
        data_dir = os.path.join(directory, "data_nosat")

    pattern = re.compile(r"^(.*?)(\d{4})\.tif$")
    stacks = {}

    for fname in os.listdir(data_dir):
        match = pattern.match(fname)
        if match:
            prefix = match.group(1)
            index = int(match.group(2))
            fullpath = os.path.join(data_dir, fname)
            stacks.setdefault(prefix, []).append((index, fullpath))

    # create stacks
    for prefix, images in stacks.items():

        images.sort(key=lambda x: x[0])

        stack = [tifffile.imread(p) for _, p in images]
        stack = np.stack(stack, axis=0)

        stack_path = os.path.join(data_dir, f"{prefix}stack.tif")

        tifffile.imwrite(
            stack_path,
            stack,
            imagej=True,
            resolution=(1/pixel_size_um, 1/pixel_size_um),
            metadata={
                "unit": "um",
                "spacing": pixel_size_um
            }
        )

        # REMOVE individual slice images
        for _, p in images:
            os.remove(p)


            
for filename in files:
    if filename.endswith(".tif"):
        image_path = os.path.join(directory, filename)
        os.remove(image_path) 
